In [15]:
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer
import numpy as np

In [11]:
model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [2]:
q1 = 'Can I still join the course after the start date?'
v1 = model.encode(q1)

In [4]:
v1.shape

(384,)

In [6]:
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)

In [7]:
# we compare the query against the document using dot product
v1.dot(dv)

np.float32(0.32332397)

In [ ]:
# another query, less similar
q2 = 'How to install Docker on Windows?'
v2 = model.encode(q2)
v2.dot(dv)

np.float32(0.019730574)

Cosine similarity

The all-MiniLM-L6-v2 model outputs normalized vectors - vectors with unit length. When both vectors are normalized, the dot product equals cosine similarity. That's why the model documentation says it "uses cosine similarity."

Cosine similarity measures the angle between two vectors, ignoring their length:

- 1.0 = same direction (similar)
- 0.0 = perpendicular (unrelated)
- -1.0 = opposite direction (opposite meaning)
Formally, if theta is the angle between two vectors, cosine similarity is cos(theta):

- cos(0) = 1 - vectors point in the same direction
- cos(90) = 0 - vectors are perpendicular
- cos(180) = -1 - vectors point in opposite directions

Because our vectors are normalized, the dot product gives us cosine similarity directly. This is why we can use v1.dot(dv) to compare texts.

### Embedding Our Dataset

### Generating embeddings

In [ ]:
from ingest import load_faq_data
# downloading the data
documents = load_faq_data()

In [13]:
# we need to combine the question and answer into a single text for embedding
texts = []

for doc in documents:
    text = doc['question'] + ' ' + doc['answer']
    texts.append(text)

In [14]:
batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/25 [00:00<?, ?it/s]

1208

In [ ]:
# We turn them into a 2-dimensional array (matrix) where

# rows are documents (vectors)
# columns are dimenstions of the vectors

X = np.array(vectors) # shape (1208, 384)

### Vector search
https://github.com/DataTalksClub/llm-zoomcamp/blob/main/02-vector-search/lessons/04-vector-search.md

In [ ]:
query = 'Can I still join the course after the start date?'

# we encode the query into a vector
v_query = model.encode(query) 

# we compute the similarity between the query and all documents using dot product
scores = X.dot(v_query)

# we find the index of the most similar document
idx = np.argmax(scores)
idx, scores[idx]

# we can now retrieve the most similar document using the index
documents[idx]

{'id': '3f1424af17',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: Can I still join the course after the start date?',
 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}

In [ ]:
top5 = np.argsort(scores)[-5:] # indices of the top 5 most similar documents
top5 = top5[::-1] # reverse to get the most similar first
scores[top5] # scores of the top 5 most similar documents


array([0.762941  , 0.7579372 , 0.7192131 , 0.6536311 , 0.56009984],
      dtype=float32)

In [20]:
for idx in top5:
    print(scores[idx])
    print(documents[idx])
    print()

0.762941
{'id': '3f1424af17', 'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}

0.7579372
{'id': '2d8b16c2a0', 'course': 'mlops-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute."}

0.7192131
{'id': '41aabbd7c5', 'course': 'machine-learning-zoomcamp', 'section': 'General Course-Related 

This is vector search in its simplest form: embed the query, compute dot products with all documents, return the ones with the highest scores.

Doing this manually with numpy works fine for small datasets. For larger ones, you'd want a search library that handles filtering and ranking. We'll use special libraries for that.

### 05 - Vector Search with minsearch
https://github.com/DataTalksClub/llm-zoomcamp/blob/main/02-vector-search/lessons/05-minsearch-vector.md

In [ ]:
from minsearch import VectorSearch 

vindex = VectorSearch(keyword_fields=['course'])
vindex.fit(X, documents) # we build the index

In [ ]:
query = 'I just discovered the course. Can I still join it?'
query_vector = model.encode(query)

results = vindex.search(query_vector, num_results=5) # we search for the most similar documents
results[0]

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

In [ ]:
results = vindex.search(
    query_vector,
    filter_dict={'course': 'llm-zoomcamp'},
    num_results=5
)
results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.'},
 {'id': 'bd31146b0e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the co

### 06-RAG with Vector Search

In [32]:
from dotenv import load_dotenv
from ingest import load_faq_data, build_index
load_dotenv()

from openai import OpenAI
import os


model_name = "models/gemini-3.1-flash-lite"

client = OpenAI(
    api_key=os.getenv('GEMINI_API_KEY'),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)


In [34]:
documents = load_faq_data()
index = build_index(documents)

In [38]:
from rag_helper import RAGBase, RAGVector

assistant = RAGBase(
    index=index,
    llm_client=client,
)

In [39]:
query = 'I just discovered the course. Can I still join it?'
assistant.rag(query)

'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'

In [ ]:
# class RAGVector(RAGBase):

#     def __init__(self, embedder, **kwargs):
#         super().__init__(**kwargs)
#         self.embedder = embedder

#     def search(self, query, num_results=5):
#         query_vector = self.embedder.encode(query)
#         filter_dict = {'course': self.course}

#         return self.index.search(
#             query_vector,
#             num_results=num_results,
#             filter_dict=filter_dict
#         )

In [41]:


vector_assistant = RAGVector(
    embedder=model,
    index=vindex,
    llm_client=client,
)

In [42]:
vector_assistant.rag('the program has already begun, can I still sign up?')

'Yes, you can still join. If you want to receive a certificate, you need to submit your project while submissions are still being accepted.'

In [46]:
vector_assistant.rag('Do I need Python skills?')

"I don't know."